In [ ]:
%load_ext autoreload
%autoreload 2
import os
from pathlib import Path
from datetime import datetime
from causaltc.data_collection.aurora_snapshot import (
    AuroraSnapshot,
    plot_snapshot,
    plot_snapshot_diff,
)
from causaltc.bogus_model.kwon_cheong_bogus import (
    BogusVortex,
    plot_bogus,
)
from causaltc.data_collection.best_track import (
    BestTrack,
)
from causaltc.data_collection.datetime_util import datetime_to_atcf_str

os.environ["CUDA_VISIBLE_DEVICES"] = "3"  # Must be done before any torch imports

PROJECT_SPACE="/projects/aurora"
# pressure levels.
levels = (1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 150, 100, 50)


In [ ]:

#
#   select a storm name / start_time pair
#
storm_dict = {
    'freddy' : {"best_track_file": Path("/projects/aurora/best-tracks/southern/bsh2023/bsh112023.dat")},
    'judy' : {"best_track_file": Path("/projects/aurora/best-tracks/southern/bsh2023/bsh152023.dat")},
    'kevin' : {"best_track_file": Path("/projects/aurora/best-tracks/southern/bsh2023/bsh162023.dat")},
    'yasa' : {"best_track_file": Path("/projects/aurora/best-tracks/southern/bsh2021/bsh052021.dat")},
    'hikaa' : {"best_track_file":Path("/projects/aurora/best-tracks/indian/bio2019/bio032019.dat")},
    'otis' :  {"best_track_file": Path("/projects/aurora/best-tracks/cepac/bep2023/bep182023.txt")},
    'rai': {"best_track_file":Path("/projects/aurora/best-tracks/westpac/bwp2021/bwp282021.dat")},
    'maria':{"best_track_file": Path("/projects/aurora/best-tracks/atlantic/bal2017/bal152017.txt")},
    'mawar':{"best_track_file": Path("/projects/aurora/best-tracks/westpac/bwp2023/bwp022023.dat")}
}

# storm_name = "freddy"
# start_time = datetime(year=2023, month=2, day=5, hour=12)

# storm_name = "judy"
# start_time = datetime(year=2023, month=2, day=22, hour=6)

# storm_name = "kevin"
# start_time = datetime(year=2023, month=3, day=2, hour=6)

# storm_name = "yasa"
# start_time = datetime(year=2020, month=12, day=15, hour=6)

# storm_name = "hikaa"
# start_time = datetime(year=2019, month=9, day=23, hour=6)

# storm_name = "otis"
# start_time = datetime(year=2023, month=10, day=21, hour=6)

# storm_name = "maria"
# # start_time = datetime(year=2017, month=9, day=17, hour=6)
# start_time = datetime(year=2017, month=9, day=18, hour=18)

storm_name = "mawar"
start_time  = datetime(year=2023, month=5, day=12, hour=12) # achieved TS name
start_time = datetime(year=2023, month=5, day=26, hour=6) # near peak

assert storm_name in storm_dict

bt = BestTrack(str(storm_dict[storm_name]["best_track_file"])) 


import numpy as np
import xarray as xr

from aurora import Batch, Metadata

assert start_time.hour > 0
hours_index = {0:0, 6:1, 12:2, 18:3}
print(f"start_time : {start_time} hour {start_time.hour} has index {hours_index[start_time.hour]}")

from huggingface_hub import hf_hub_download

# Download the static variables from HuggingFace.
#   HuggingFace automatically caches its data, so this will only perform the download if the file is not 
#   found in cache_dir.
static_path = hf_hub_download(
    repo_id="microsoft/aurora",
    filename="aurora-0.1-static.pickle",
    cache_dir="/projects/aurora/aurora_data"
)
print(f"Static variables downloaded: {static_path}")

def aurora_start_data(start_time:datetime):
    date = start_time
    date_str = f"{date.year:04d}{date.month:02d}{date.day:02d}"
    surf_file = "hres_surf_" + date_str + ".grb"
    atmos_file = "hres_atmos_" + date_str + "_fh00_06_12_18.grb"
    return surf_file, atmos_file

surf_data_file, atmos_data_file = aurora_start_data(start_time)

DATA_PATH = Path("/projects/aurora/ecmwf_ifs_01deg")
print(f"looking for files: {surf_data_file} and {atmos_data_file} in {DATA_PATH}")

def open_grib(path: Path) -> xr.Dataset:
    """Open a GRIB file via cfgrib without using persistent .idx files."""
    if not path.exists():
        raise FileNotFoundError(f"GRIB file does not exist: {path}")
    if not path.is_file():
        raise FileNotFoundError(f"Path is not a file: {path}")
    return xr.open_dataset(
        path,
        engine="cfgrib",
        backend_kwargs={"indexpath": ""},
    )
surf_ds = open_grib(DATA_PATH / surf_data_file)
atmos_ds = open_grib(DATA_PATH / atmos_data_file)

import torch
import pickle

def load_surf(ds: xr.Dataset, v_in_file: str, start_hour:int) -> torch.Tensor:
    """
    Load a surface variable for the first two time steps (00 and 06 UTC).

    Returns a tensor with shape (batch, time, latitude, longitude).
    """
    assert hours_index[start_hour] > 0
    time_inds = [hours_index[start_hour], hours_index[start_hour]-1]
    da = ds[v_in_file].isel(time=time_inds).transpose("time", "latitude", "longitude")
    data = da.values.astype(np.float32)[None]   # add batch dimension
    return torch.from_numpy(data)

def load_atmos(ds: xr.Dataset, v: str, start_hour:int) -> torch.Tensor:
    """
    Load an atmospheric variable for the first two time steps (00 and 06 UTC).

    Returns a tensor with shape
    (batch, time, isobaricInhPa, latitude, longitude).
    """
    assert hours_index[start_hour] > 0
    time_inds = [hours_index[start_hour], hours_index[start_hour]-1]
    da = ds[v].sel(isobaricInhPa=list(levels))
    da = da.isel(time=time_inds)
    da = da.transpose("time", "isobaricInhPa", "latitude", "longitude")
    data = da.values.astype(np.float32)[None]   # add batch dimension
    return torch.from_numpy(data)

batch = Batch(
    surf_vars={
        "2t": load_surf(surf_ds, "t2m", int(start_time.hour)),
        "10u": load_surf(surf_ds, "u10", start_time.hour),
        "10v": load_surf(surf_ds, "v10", start_time.hour),
        "msl": load_surf(surf_ds, "msl", start_time.hour),
    },
    # Static vars added after regridding.
    static_vars={},
    atmos_vars={
        "t": load_atmos(atmos_ds, "t", start_time.hour),
        "u": load_atmos(atmos_ds, "u", start_time.hour),
        "v": load_atmos(atmos_ds, "v", start_time.hour),
        "q": load_atmos(atmos_ds, "q", start_time.hour),
        "z": load_atmos(atmos_ds, "z", start_time.hour),
    },
    metadata=Metadata(
        lat=torch.from_numpy(atmos_ds.latitude.values.astype(np.float32)),
        lon=torch.from_numpy(atmos_ds.longitude.values.astype(np.float32)),
        time=(start_time,),
        atmos_levels=levels,
    ),
)

print("PRIOR TO REGRIDDING:")
print(f"Aurora start batch ready for start time {batch.metadata.time[-1]}")
print(f"metadata = {batch.metadata}")
print("Surface shapes:")
for k, v in batch.surf_vars.items():
    print(f"  {k}: {tuple(v.shape)}")

print("Atmospheric shapes:")
for k, v in batch.atmos_vars.items():
    print(f"  {k}: {tuple(v.shape)}")

# Regrid to 0.1 degree resolution.
# Note: convenience function, not optimized for speed or accuracy.
batch = batch.regrid(res=0.1)

# Load static variables.
with open(static_path, "rb") as f:
    static_vars = pickle.load(f)

batch.static_vars = {
    k: torch.from_numpy(np.asarray(v, dtype=np.float32))
    for k, v in static_vars.items()
}

print("REGRIDDING COMPLETE.")
print(f"Aurora start batch ready for start time {batch.metadata.time[-1]}")
print(f"metadata = {batch.metadata}")
print("Surface shapes:")
for k, v in batch.surf_vars.items():
    print(f"  {k}: {tuple(v.shape)}")

print("Atmospheric shapes:")
for k, v in batch.atmos_vars.items():
    print(f"  {k}: {tuple(v.shape)}")

In [ ]:
valid_time = start_time
storm_loc = bt.get_location(valid_time)

snapshot = AuroraSnapshot(start_time, valid_time)
snapshot.collect_data_from_step(batch=batch, box_center=storm_loc, box_size=10.0, verbose=True)
snapshot.compute_storm_data(verbose=True)
snapshot.storm_found(verbose=True)

def deepcopy_batch(batch: Batch) -> Batch:
    return Batch(
        surf_vars={k: v.detach().clone() for k, v in batch.surf_vars.items()},
        static_vars={k: v.detach().clone() for k, v in batch.static_vars.items()},
        atmos_vars={k: v.detach().clone() for k, v in batch.atmos_vars.items()},
        metadata=Metadata(
            lat=batch.metadata.lat.detach().clone(),
            lon=batch.metadata.lon.detach().clone(),
            time=tuple(batch.metadata.time),
            atmos_levels=tuple(batch.metadata.atmos_levels),
            rollout_step=batch.metadata.rollout_step,
        ),
    )

batch2 = deepcopy_batch(batch=batch)

In [ ]:
bg = BogusVortex(snapshot=snapshot, best_track=bt, target_time=valid_time)
bg.build(verbose=True)
plot_bogus(bg, f"bogus_vortex_{storm_name}_valid{datetime_to_atcf_str(valid_time)}.pdf")

bg.apply(batch=batch2)

In [ ]:
snapshot2 = AuroraSnapshot(start_time, valid_time)
snapshot2.collect_data_from_step(batch=batch2, box_center=storm_loc, box_size=10.0, verbose=True)
snapshot2.compute_storm_data(verbose=True)
snapshot2.storm_found(verbose=True)
plot_snapshot(snapshot, f"{storm_name}-pre-bogus-snapshot.pdf")
plot_snapshot(snapshot2,f"{storm_name}-post-bogus-snapshot.pdf")
plot_snapshot_diff(snapshot, snapshot2, zoom_deg=5.0, filename=f"{storm_name}-bogus-snapshot-diff.pdf") 

In [ ]:
from aurora import AuroraHighRes

model_data_dir = Path("/projects/aurora/aurora_models/models--microsoft--aurora/snapshots/d8753da77552eab0f27150c0efb91cb3acb425d5")
model_checkpoint_file = "aurora-0.1-finetuned.ckpt"

model = AuroraHighRes()
model.load_checkpoint_local(model_data_dir / model_checkpoint_file, strict=False)

In [ ]:
from causaltc.data_collection.aurora_data_history import (
    SingleAuroraDataHistory,
    plot_snapshots
)

rollout_length = 12

dh = SingleAuroraDataHistory.generate(best_track=bt, model=model, start_time=start_time, rollout_length=rollout_length, batch=batch, box_half_size=10.0, intervened=False, verbose=True, save=False)
comparison = dh.to_comparison_dataframe(best_track=bt, verbose=True)
plot_snapshots(dh, f"rollout_{storm_name}_start_{datetime_to_atcf_str(start_time)}_len{rollout_length}.pdf", best_track=bt)

In [ ]:
dh2 = SingleAuroraDataHistory.generate(best_track=bt, model=model, start_time=start_time, rollout_length=rollout_length, batch=batch2, box_half_size=10.0, intervened=True, verbose=True, save=False)
comparison = dh2.to_comparison_dataframe(best_track=bt, verbose=True)
plot_snapshots(dh2, f"rollout_{storm_name}_bogus_vortex_start_{datetime_to_atcf_str(start_time)}_len{rollout_length}.pdf", best_track=bt)

In [ ]:
dh.save("aurora_rollout_control", verbose=True)

In [ ]:
from pathlib import Path
dh.save(Path("aurora_rollout_control"), verbose=True)
dh2.save(Path("aurora_rollout_bogus"), verbose=True)